In [9]:
import os
from openai import OpenAI

# NVIDIA Build API client
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY")
)

MODEL = "meta/llama-3.1-70b-instruct"

In [3]:
# ================================
# Shared Memory / Message Bus
# ================================

class SharedMemory:
    def __init__(self):
        self.data = {}

    def update(self, agent, result):
        self.data[agent] = result

    def get(self, agent):
        return self.data.get(agent, "")

    def all(self):
        return self.data

In [4]:
# ================================
# Base Agent
# ================================

class Agent:
    def __init__(self, name, role):
        self.name = name
        self.role = role

    def run(self, goal, memory):

        context = str(memory.all())

        prompt = f"""
You are {self.name}.

Role: {self.role}

Goal:
{goal}

Shared Memory:
{context}

Perform your role and produce output.
"""

        completion = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": self.role},
                {"role": "user", "content": prompt}
            ],
            temperature=0.4,
            max_tokens=1200
        )

        return completion.choices[0].message.content

In [5]:
# ================================
# Define Agents
# ================================

search_agent = Agent(
    "Search Agent",
    "Find relevant EV market data for India 2025 including growth trends, companies, and government policies."
)

analyst_agent = Agent(
    "Analyst Agent",
    "Analyze the search results, extract insights, trends, statistics and summarize them."
)

writer_agent = Agent(
    "Writer Agent",
    "Write a structured market research report using the analyst insights."
)

qa_agent = Agent(
    "QA Agent",
    "Check the report for factual consistency, clarity, and professional tone. Fix issues if necessary."
)

In [6]:
# ================================
# Orchestrator
# ================================

def run_pipeline(goal):

    agents = [
        search_agent,
        analyst_agent,
        writer_agent,
        qa_agent
    ]

    memory = SharedMemory()

    for agent in agents:

        print(f"\nRunning {agent.name}...\n")

        result = agent.run(goal, memory)

        memory.update(agent.name, result)

    final_report = memory.get("QA Agent")

    return final_report

In [7]:
# ================================
# Run System
# ================================

goal = "Write a comprehensive market report on EV industry trends in India for 2025."

report = run_pipeline(goal)

print("\nFINAL REPORT\n")
print(report)


Running Search Agent...


Running Analyst Agent...


Running Writer Agent...


Running QA Agent...


FINAL REPORT

**Comprehensive Market Report on EV Industry Trends in India for 2025**

**Executive Summary:**

The Electric Vehicle (EV) market in India has witnessed significant growth in 2025, driven by government policies, increasing demand for sustainable transportation, and declining battery costs. This report provides an overview of the Indian EV market, highlighting growth trends, key players, and government initiatives.

**Market Size and Growth Trends:**

* The Indian EV market is expected to reach 1.3 million units in 2025, growing at a Compound Annual Growth Rate (CAGR) of 43% from 2020 to 2025. (Source: Society of Manufacturers of Electric Vehicles (SMEV))
* The EV market share in India is projected to increase from 1.3% in 2020 to 5.5% in 2025. (Source: International Energy Agency (IEA))
* The two-wheeler segment dominates the Indian EV market, accounting for 70% of total 